# Notebook 08 — Interactive Charts with Plotly

Everything up to this point used matplotlib and seaborn for static charts. This notebook rebuilds the key visuals using Plotly, which produces interactive HTML that you can hover over, zoom, and filter.

Five charts in total:

1. **National performance over time** — the headline trend with the 95% target line
2. **Seasonal monthly profile** — average performance by calendar month
3. **Regional distribution** — box plot showing spread across trusts by region (latest year)
4. **Volume vs performance scatter** — one dot per trust, size indicates volume, local trust highlighted
5. **Performance band trend** — stacked bar showing how the mix of trusts has shifted over time

All charts are exported as `.html` files to `outputs/figures/` so they can be shared or embedded.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

national = pd.read_csv("../data/processed/dashboard_national.csv", parse_dates=["period"])
trust = pd.read_csv("../data/processed/dashboard_trust.csv", parse_dates=["quarter_start_date"])

# strip trailing spaces from region names
trust["region_short"] = trust["region_short"].str.strip()

print(f"National: {len(national)} rows ({national['period'].min().strftime('%b %Y')} to {national['period'].max().strftime('%b %Y')})")
print(f"Trust: {len(trust)} rows, {trust['code'].nunique()} trusts, {trust['financial_year'].nunique()} years")

National: 85 rows (Apr 2019 to Apr 2026)
Trust: 5062 rows, 258 trusts, 6 years


## Chart 1 — National Type 1 performance over time

The headline chart. Type 1 is major A&E — the 95% target applies here. We shade the COVID period (March 2020 to June 2021) since the drop in attendances during that window makes the performance numbers hard to interpret normally.

In [2]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=national["period"],
    y=national["pct_4hr_type1_pct"],
    mode="lines",
    name="Type 1 % seen within 4hrs",
    line=dict(color="#005EB8", width=2),
    hovertemplate="%{x|%b %Y}: %{y:.1f}%<extra></extra>"
))

# 95% target line
fig.add_hline(
    y=95, line_dash="dash", line_color="red", line_width=1.5,
    annotation_text="95% target", annotation_position="top right",
    annotation_font_color="red"
)

# COVID shading
fig.add_vrect(
    x0="2020-03-01", x1="2021-06-01",
    fillcolor="orange", opacity=0.12, line_width=0,
    annotation_text="COVID-19", annotation_position="top left"
)

fig.update_layout(
    title="NHS A&E Type 1 Performance: % seen within 4 hours (April 2019 – April 2026)",
    xaxis_title="",
    yaxis_title="% seen within 4 hours",
    yaxis=dict(range=[50, 100], ticksuffix="%"),
    plot_bgcolor="white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0")

fig.write_html("../outputs/figures/plotly_01_national_performance.html")
fig.show()

## Chart 2 — Seasonal monthly profile

Averaging performance by calendar month across all years (excluding COVID 2020-21 to avoid distorting the seasonal pattern). If winter months consistently underperform, that tells us demand-side pressure is a real factor — even if it doesn't fully explain the overall decline.

In [3]:
# exclude COVID year from the seasonal average
non_covid = national[~national["financial_year"].isin(["2020-21"])].copy()

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

monthly_avg = (
    non_covid.groupby("month_name")["pct_4hr_type1_pct"]
    .mean()
    .reindex(month_order)
    .reset_index()
)

fig = px.bar(
    monthly_avg,
    x="month_name",
    y="pct_4hr_type1_pct",
    title="Average Type 1 Performance by Month (excl. COVID year 2020-21)",
    labels={"pct_4hr_type1_pct": "Avg % seen within 4hrs", "month_name": ""},
    color="pct_4hr_type1_pct",
    color_continuous_scale=["#d73027", "#fee08b", "#1a9850"],
    range_color=[70, 88],
    text_auto=".1f"
)

fig.add_hline(
    y=95, line_dash="dash", line_color="red", line_width=1.5,
    annotation_text="95% target", annotation_position="top right",
    annotation_font_color="red"
)

fig.update_layout(
    plot_bgcolor="white",
    yaxis=dict(range=[60, 100], ticksuffix="%"),
    coloraxis_showscale=False
)
fig.update_traces(textposition="outside")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0")

fig.write_html("../outputs/figures/plotly_02_seasonal_profile.html")
fig.show()

## Chart 3 — Regional performance distribution (latest year)

Box plot showing the spread of Type 1 performance across individual trusts, grouped by NHS England region. The box shows the interquartile range, the line is the median, and dots are outliers. This is more informative than a simple regional average because it shows whether variation within regions is high or low.

In [4]:
latest_year = trust["financial_year"].max()
latest = trust[
    (trust["financial_year"] == latest_year) &
    trust["pct_4hr_type1_pct"].notna() &
    trust["region_short"].notna()
].copy()

# shorten region labels for readability
latest["region_label"] = (
    latest["region_short"]
    .str.replace("NHS England ", "", regex=False)
    .str.replace(" And ", " & ", regex=False)
)

# sort by median performance
region_order = (
    latest.groupby("region_label")["pct_4hr_type1_pct"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    latest,
    x="region_label",
    y="pct_4hr_type1_pct",
    title=f"Type 1 Performance by Region — {latest_year} (individual trusts)",
    labels={"pct_4hr_type1_pct": "% seen within 4hrs", "region_label": ""},
    color="region_label",
    points="outliers",
    category_orders={"region_label": region_order},
    hover_data=["name"]
)

fig.add_hline(
    y=95, line_dash="dash", line_color="red", line_width=1.5,
    annotation_text="95% target", annotation_position="top right",
    annotation_font_color="red"
)

fig.update_layout(
    plot_bgcolor="white",
    yaxis=dict(ticksuffix="%"),
    showlegend=False
)
fig.update_xaxes(showgrid=False, tickangle=-20)
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0")

fig.write_html("../outputs/figures/plotly_03_regional_boxplot.html")
fig.show()

## Chart 4 — Volume vs performance scatter (latest year)

Does a higher patient volume mean worse performance? This scatter plots each trust as a dot — x axis is total Type 1 attendances for the year, y axis is average performance. Colour shows region. The local trust (North West Anglia — Hinchingbrooke) is highlighted and labelled.

If there's a clear downward trend it would suggest busier trusts are systematically struggling. If the scatter is all over the place, volume alone isn't the driver.

In [5]:
trust_agg = (
    latest
    .groupby(["code", "name", "region_label", "is_local_trust"])
    .agg(
        total_type1=("type1_attendances", "sum"),
        avg_perf=("pct_4hr_type1_pct", "mean")
    )
    .reset_index()
    .dropna(subset=["avg_perf", "total_type1"])
)

trust_agg["label"] = trust_agg.apply(
    lambda r: r["name"] if r["is_local_trust"] == 1 else "", axis=1
)

fig = px.scatter(
    trust_agg,
    x="total_type1",
    y="avg_perf",
    color="region_label",
    hover_name="name",
    text="label",
    title=f"Trust Volume vs Performance — {latest_year}",
    labels={
        "total_type1": "Type 1 attendances (annual total)",
        "avg_perf": "Avg % seen within 4hrs",
        "region_label": "Region"
    }
)

# make local trust stand out
local_trust = trust_agg[trust_agg["is_local_trust"] == 1]
if not local_trust.empty:
    fig.add_trace(go.Scatter(
        x=local_trust["total_type1"],
        y=local_trust["avg_perf"],
        mode="markers",
        marker=dict(symbol="star", size=16, color="gold", line=dict(color="black", width=1)),
        name="North West Anglia (local)",
        hovertext=local_trust["name"],
        hoverinfo="text"
    ))

fig.add_hline(
    y=95, line_dash="dash", line_color="red", line_width=1.5,
    annotation_text="95% target", annotation_position="top right",
    annotation_font_color="red"
)

fig.update_traces(textposition="top center", selector=dict(mode="markers+text"))
fig.update_layout(
    plot_bgcolor="white",
    yaxis=dict(ticksuffix="%")
)
fig.update_xaxes(showgrid=True, gridcolor="#e0e0e0")
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0")

fig.write_html("../outputs/figures/plotly_04_scatter_volume_performance.html")
fig.show()

## Chart 5 — Performance band distribution over time

Stacked bar showing what share of trusts fell into each performance band each year. The green band at the top (>= 95%) shrinking over time tells a clear story about system-wide decline — it's not just a few struggling trusts pulling the average down, it's the whole distribution shifting.

In [6]:
band_order = ["1. >= 95% (target)", "2. 85-94%", "3. 75-84%", "4. < 75% (severe)"]
band_colors = {
    "1. >= 95% (target)": "#1a9850",
    "2. 85-94%": "#fee08b",
    "3. 75-84%": "#fc8d59",
    "4. < 75% (severe)": "#d73027"
}

band_counts = (
    trust[trust["performance_band"].notna()]
    .groupby(["financial_year", "performance_band"])
    .size()
    .reset_index(name="count")
)

totals = band_counts.groupby("financial_year")["count"].sum().reset_index(name="total")
band_counts = band_counts.merge(totals, on="financial_year")
band_counts["pct"] = (band_counts["count"] / band_counts["total"] * 100).round(1)
band_counts = band_counts[band_counts["performance_band"].isin(band_order)]

fig = px.bar(
    band_counts,
    x="financial_year",
    y="pct",
    color="performance_band",
    title="Share of NHS Trusts by Performance Band — 2019-20 to 2024-25",
    labels={"pct": "% of trusts", "financial_year": "", "performance_band": ""},
    category_orders={"performance_band": band_order},
    color_discrete_map=band_colors,
    barmode="stack",
    text_auto=".0f"
)

fig.update_layout(
    plot_bgcolor="white",
    yaxis=dict(ticksuffix="%", range=[0, 105]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, traceorder="normal")
)
fig.update_traces(texttemplate="%{y:.0f}%", textposition="inside")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0")

fig.write_html("../outputs/figures/plotly_05_performance_band_trend.html")
fig.show()

## Done

Five interactive charts exported to `outputs/figures/`:

| File | Chart |
|------|-------|
| `plotly_01_national_performance.html` | National Type 1 trend with COVID shading |
| `plotly_02_seasonal_profile.html` | Monthly average performance (seasonal) |
| `plotly_03_regional_boxplot.html` | Regional distribution, latest year |
| `plotly_04_scatter_volume_performance.html` | Volume vs performance scatter |
| `plotly_05_performance_band_trend.html` | Performance band stacked bar |

Open any `.html` file in a browser to get the full interactive version — hover, zoom, click legend items to toggle series.